# **決定木のアルゴリズム (C4.5, C5.0)**

ここでは，書籍第3章3.4節「決定木のアルゴリズム」でとりあげた3つの代表的な決定木アルゴリズム (CART, CHAID, ID3) に追加して，C4.5 と C5.0 を紹介します．

## **C4.5による分類木アルゴリズム**

R. Quinlan によって提案された C4.5 は，同じく Quinlan による ID3 アルゴリズムの改良版です．

ID3 では，データの分割基準（指標）として情報利得を用いますが，例えば郵便番号や商品 ID のような値の種類が多い特徴量が選ばれやすくなるという特性がありました．また，ID3 は数値データ（連続変数）や欠損値の扱いに対応しておらず，作成される決定木が複雑になりすぎて，学習データにだけ適合してしまう過学習が起こりやすいという問題点もありました．C4.5 では，これらの問題点を次のように改善しています．

- データの分割基準として情報利得を正規化した「ゲイン比」を導入し，値の種類が多い特徴量が過剰に選ばれないようにする

- 数値データでは，最適な分割のための閾値を自動的に決める

- 欠損値については，既存データの分布に基づいて処理する

- 悲観的誤差推定（詳しくは**Step 5**で解説します）に基づき，作成した決定木を必要に応じてプルーニングし，過学習を防ぐ

以下，C4.5 アルゴリズムを用いた分類モデルの構築について説明します．

<br>

#### **■Step 1.データの準備**

まず，分析対象となるデータを次の形式で準備します．

$$(X_{11},X_{12},...,X_{1p},y_{1}),\ (X_{21},X_{22},...,X_{2p},y_{2}),\ ...,\ (X_{N1},X_{N2},...,X_{Np},y_{N})$$

ここで，$X_{ij}$ は $i( = 1,2,\cdots,N)$ 番目のデータ点の特徴量 $j( = 1,2,\cdots,p)$ の値，$y_{i}$ はターゲット変数，すなわちデータ点が属するクラスです．このように，$N$ 個のデータ点に対して $p$ 個の特徴量をもつデータセットが準備されます．

C4.5 は，欠損値（値が記録されていないデータ）や連続変数（年齢や収入のような数値データ）も柔軟に処理します．欠損値に対しては，データの分布に基づいて欠損値を補間します．例えば，ほとんどの人が「男性」であれば，欠損値があるデータも「男性」と仮定する場合があります．また，連続変数に対しては，データを適切な閾値で分割します．

<br>

#### **■Step 2.分類木の初期設定**

分類木を構築するにあたり，各ノードにおいてデータ集合をどのように分割するかを評価するための基準を定めます．C4.5 による分類木では，ID3 と同様，エントロピーを不純度の指標として用います（→書籍3.4.4項）．

<br>

#### **■Step 3. 子ノードの生成**

特徴量 $X_{ij}$ による分割点の評価のため，条件付きエントロピーを計算します．さらに，その条件付きエントロピーを用いて特徴量ごとに情報利得を求めます．ここまでは ID3 と同様です（→書籍3.4.4項）．

ID3 では情報利得が最大の特徴量を選んでデータを分割しますが，この方法には「とり得る値の種類が多い特徴量ほど情報利得が過大に評価されやすい」という問題があります．C4.5 では，これを補正するために「ゲイン比」を分割基準として導入しています．ゲイン比は，情報利得を「分割情報量」で割ることで正規化した量です．分割情報量は，データの分割そのものがどれだけ多くの情報を必要とするかを測る指標であり，値の種類が多い特徴量ほど大きな値をとります．情報利得をこの分割情報量で割ることで，値の種類が多い特徴量が不当に有利にならないよう補正されます．

特徴量 $X_{ij}$ に基づくゲイン比 $\mathrm{GR}(S,X_{ij})$（$S$ は全てのデータ点の集合）は，次の式で表されます．

$$\mathrm{GR}(S,X_{ij}) = \frac{\mathrm{Gain}(S,X_{ij})}{\mathrm{IV}(S,X_{ij})}$$

ここで，$\mathrm{Gain}(S,X_{ij})$ は情報利得（→書籍3.4.4項）です．また，$\mathrm{IV}(S,X_{ij})$ は分割情報量で，次の式で計算されます．

$$\mathrm{IV}(S,X_{ij}) = - \sum_{v \in V(X_{ij})}^{}\frac{|S_{v}|}{|S|}\log_{2}\frac{|S_{v}|}{|S|}$$

この式で，$V(X_{ij})$ は特徴量 $X_{ij}$ のとり得る値（カテゴリ）の集合，$S_{v}$ は $X_{ij} = v$ を満たすデータ点の部分集合を表します．ゲイン比 $\mathrm{GR}(S,X_{ij})$ が最大の特徴量 $X_{ij}$ を，そのノードのデータを子ノードへ分割する際の基準として選びます．

こうして選ばれた特徴量 $X_{ij}$ における分割点は，次のように決めます．

1.  カテゴリ変数の特徴量の場合は，各カテゴリ値を候補としてゲイン比が最大となる分割点を決定します．連続値の特徴量の場合は，データを昇順に並べ，隣接する値の境界を分割候補として評価し，情報利得やゲイン比が最大となる分割点（閾値）を決定します．

2.  選ばれた分割点でデータを2つに分け，そのグループを新しい子ノードとして扱います．

3.  その後，各子ノードに対して再度同じ操作を行い，さらに細かくデータを分割していきます．

4.  上記の手順を繰り返して，木全体を構築していきます．

<br>

#### **■Step 4.終了条件を満たすまでの再帰的な木の成長**

C4.5 アルゴリズムでは，次のいずれかの条件を満たすと分割を停止し，そのノードをリーフノードとして扱います．

- すべてのデータ点が同じターゲットクラスに属する場合：すべてのデータが同じクラスに分類されている場合，そのノードはリーフノードになります（ID3 と同じ条件です）．

- 特徴量が残っていない場合：ルートノードから現在のノードに至るまでにすべての特徴量が使われ，これ以上分割できない場合は，そのノードに含まれるデータ点の多数決でクラスを決定します（ID3 と同じ条件です）．

- 分割によって得られる情報利得またはゲイン比が指定された最小値以下の場合：分割してもほとんど分類性能の改善が期待できない場合は，そのノードはリーフノードとされます（C4.5 で追加された条件です）．

ただし，C4.5 では，これらの終了条件だけでなく，分割後に行う悲観的誤差推定に基づくプルーニング（剪定）が重要な役割を果たし，過学習を防ぐ点も特徴です．

<br>

#### **■Step 5. プルーニングによる過学習の防止**

C4.5 では，過学習を防ぐために**悲観的誤差推定** (pessimistic error estimation) という手法を用いてプルーニング（剪定）を行います．この方法は，トレーニングデータ上の単純な誤分類率を鵜呑みにするのではなく，将来のデータに対して誤差が大きくなる可能性を考慮し，より保守的（悲観的）な誤差率を見積もる点が特徴です．

以下に，各ノードにおける誤差率推定を含む具体的な計算の流れを示します．

ノード内の総サンプル数を $N$，クラス$k$に属するサンプル数を $N_{k}$ とすると，ノード内の誤分類数 $e$ は，次の式で表されます．

$$e = N - \max_{k}N_{k}$$

すると，トレーニングデータ上の誤分類率（観測誤分類率）は $e/N$ と表されますが，これをそのまま使用するとトレーニングデータに最適化されすぎる可能性があります．そこで C4.5 では，二項分布の信頼区間の上限を誤分類率の推定値として採用した「悲観的誤差率」を用います．具体的には，観測誤分類率 $e/N$ を真の誤分類率 $p$ の推定量とみなし，その信頼区間の上限を求め悲観的誤差率 $\widehat{e}$ とします．この上限は二項分布から求められ，導出は省略しますが次の近似式で表せます．

$$\widehat{e} = e + \frac{z^{2}}{2} + z\sqrt{\frac{e}{N}\left( 1 - \frac{e}{N} \right) + \frac{z^{2}}{4N^{2}}}$$

ただし，$z$ は信頼区間の広さを決める値です．標準正規分布において，区間の両外側に合計25%の確率が残るように境界を定めると $z = 0.69$ となり，これが約75%信頼区間に対応します．なお，Quinlan が C4.5 以前に提案した旧来の悲観的誤差率では，計算を簡略化した次の近似式が用いられていました．

$$\widehat{e} = e + 0.5$$

次に，部分木全体の悲観的誤差率を計算します．部分木 $T$ 全体の悲観的誤差率は，リーフノードの悲観的誤差数の合計を，部分木内の全サンプル数 $N_{T}$ で割ることで求めます．

$$\frac{{\widehat{E}}_{T}}{N_{T}} = \frac{\sum_{i \in \text{leaves}(T)}^{}{\widehat{e}}_{i}}{N_{T}}$$

一方，部分木 $T$ を1つのリーフにまとめた場合の悲観的誤差率は，次の通りです．

$$\frac{{\widehat{E}}_{R}}{N_{T}} = \frac{e_{T} + 0.5}{N_{T}}$$

ここで，$e_{T}$ は部分木 $T$ 全体の誤分類数です．

最後に，プルーニングを行うか否かを判断します．部分木を保持する場合と，1つのリーフに置き換える場合の悲観的誤差率を比較し，

$$\frac{{\widehat{E}}_{R}}{N_{T}} \leq \frac{{\widehat{E}}_{T}}{N_{T}}$$

を満たす場合には，部分木をプルーニングし単一のリーフノードに置き換えます．

このようにして，学習データに依存しすぎない，より汎化性能の高い決定木が得られます．

<br>

#### **■Step 6.分類木の評価**

C4.5 アルゴリズムで構築された決定木は，適切に分割されているか，未知のデータに対してどの程度の性能を発揮するかを評価する必要があります．この工程は，モデルの有効性を確認し，改善の余地を検討するための最終ステップです（→詳細は3.3節）．

## **Python で C4.5 による分類木構築**

C4.5 を用いて Titanic 乗客の生存予測モデルを作成します．使用する seaborn の Titanic データセットには，性別，階級，乗船地などのカテゴリ型特徴量と生存フラグが含まれています．

#### **■Step 0. ライブラリのインポート**

必要なライブラリをインポートします．

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split

#### **■Step 1. データの準備**

load_data() 関数は，Titanic データセットを読み込み，機械学習用の前処理を行う関数です．具体的には，特徴量として sex（性別），class（客室クラス），embarked（乗船港）を抽出し，欠損値を除外した後，これらの列をカテゴリ型に変換します．また，データを層化サンプリングに基づきトレーニング用とテスト用に分割し，その結果を返します．

In [2]:
def load_data():
    # Titanic データセットを読み込む
    df = sns.load_dataset('titanic').dropna(subset=['embarked', 'class', 'sex', 'age', 'fare'])

    # 特徴量として sex, class, embarked を抽出
    X = df[['sex', 'class', 'embarked']].copy()
    y = df['survived'].values

    # 特徴量をカテゴリ型に変換
    for col in X.columns:
        X[col] = X[col].astype('category')

    # データを層化サンプリングでトレーニング用・テスト用に分割
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)

    return X_train, X_test, y_train, y_test

#### **■Step 2.分類木の初期設定**

entropy() 関数は，ターゲット変数yに基づいてエントロピー（クラスの不確実性）を計算します．

information_gain() 関数は，データを分割したときにエントロピーがどれだけ減少するか（情報利得）を求めます．

In [3]:
# エントロピーを計算する関数
# y: ターゲット変数（例: 生存=1, 死亡=0 のラベル配列）
def entropy(y):
    counts = np.bincount(y)  # クラスごとの出現回数を計算（例: [死亡数, 生存数]）
    probs = counts / len(y)  # クラスごとの出現割合を計算（例: [0.6, 0.4]）
    # 各クラスについて -p * log2(p) を計算し，その総和を返す（p>0 の場合のみ）
    return -np.sum([p * np.log2(p) for p in probs if p > 0])

# 情報利得を計算する関数
# y: 元のターゲット変数
# groups: 分割後のサブグループのy値リスト（例: [y1, y2, y3]）
def information_gain(y, groups):
    H_before = entropy(y)  # 分割前の全体エントロピー
    # 各グループのエントロピーを計算し，サイズ比で重み付けした合計を求める
    H_after = sum(len(group) / len(y) * entropy(group) for group in groups)
    return H_before - H_after  # 情報利得 = 分割前エントロピー - 分割後エントロピー

#### **■Step 3.子ノードの生成**

split_info() 関数は，分割後のサブグループの割合に基づいて分割情報量を計算します．

gain_ratio( )関数は，情報利得を分割情報で割ることでゲイン比を算出し，過剰分割の影響を抑えます．

best_split_c45() 関数は，全特徴量の中から最もゲイン比が高い特徴量を選びます．**Step 1**でカテゴリ型に変換されたデータを前提に，カテゴリ値ごとの分割を行います．

In [4]:
# 分割情報（split info）を計算する関数
# groups: 分割後のサブグループ，y_len: 全体のサンプル数
def split_info(groups, y_len):
    sizes = [len(group) for group in groups]  # 各グループのサイズを取得
    probs = [size / y_len for size in sizes]  # 各グループのサイズ比を計算
    # 各グループについて -p * log2(p) を計算し，その総和を返す（p>0 の場合のみ）
    return -np.sum([p * np.log2(p) for p in probs if p > 0])

# ゲイン比（gain ratio）を計算する関数
# 情報利得を分割情報で割って正規化し，過剰分割を防ぐ
def gain_ratio(y, groups):
    ig = information_gain(y, groups)  # 情報利得を計算
    si = split_info(groups, len(y))   # 分割情報を計算
    return ig / si if si != 0 else 0  # ゼロ割りを避けるため，si=0 の場合は0を返す

# 最適な分割基準を決定する関数（C4.5用）
# X: 特徴量データフレーム，y: ターゲット変数
def best_split_c45(X, y):
    best_score, best_feature = -1, None  # ベストスコアとベスト特徴量を初期化
    for col in X.columns:  # 各特徴量についてループ
        # 特徴量の各カテゴリ値に基づいて y を分割
        groups = [y[X[col] == val] for val in X[col].unique()]
        if len(groups) < 2:
            continue  # 分割できない場合はスキップ
        gr = gain_ratio(y, groups)  # ゲイン比を計算
        if gr > best_score:
            best_score, best_feature = gr, col  # ベストを更新
    return best_feature  # 最もゲイン比が高い特徴量を返す

#### **■Step 4.終了条件を満たすまでの再帰的な木の成長**

build_tree_c45() 関数の内部で，終了条件を確認します．具体的には，最大深さ (max_depth) に到達した場合，ノード内のすべてのサンプルが同じクラスの場合，これ以上分割できない場合のいずれかに該当したら，それ以上の分割を行わずリーフノードを作ります．

In [5]:
def build_tree_c45(X, y, depth=0, max_depth=5):
    counts = Counter(y)  # クラスごとの出現回数を数える
    majority = counts.most_common(1)[0][0]  # 最頻クラス（多数派のクラス）を取得

    # ノード情報を辞書形式で作成
    node = {'depth': depth, 'majority': majority}

    # Step4. 終了条件の確認
    # 1. 最大深さに達した場合
    # 2. 残りデータがすべて同じクラスの場合
    if depth >= max_depth or len(set(y)) == 1:
        node['is_leaf'] = True  # リーフノードにする
        return node

    # --- 最適な分割特徴量を選択 ---
    feature = best_split_c45(X, y)
    if feature is None:
        node['is_leaf'] = True  # 分割できない場合はリーフノードにする
        return node

    # --- 子ノードの生成 ---
    node['is_leaf'] = False  # リーフノードではないことを設定
    node['feature'] = feature  # 分割に使う特徴量を保存
    node['children'] = {}  # 子ノードを格納する辞書を初期化

    # 特徴量の各カテゴリ値ごとにデータを分割し，再帰的に子ノードを作成
    for val in X[feature].unique():
        mask = X[feature] == val  # 特徴量が val のサンプルを抽出
        # 再帰的に build_tree_c45 を呼び出し，子ノードを作成
        child = build_tree_c45(X[mask], y[mask], depth + 1, max_depth)
        node['children'][val] = child  # 子ノードを登録

    return node  # 作成したノードを返す

#### **■Step 5.プルーニング（剪定）**

prune() 関数は，構築済みの分類木に対して悲観的誤差推定を用いたプルーニングを行います．具体的には，まず各ノードのすべての子がリーフノードかどうかを確認し，もしそうならば，その部分木全体の誤差と，ノード単体の誤差を悲観的誤差推定で計算します．その結果，ノード単体の誤差の方が低ければ，部分木全体をまとめてリーフノードに変換します．

pessimistic_error() 関数は，誤分類数とサンプル数から信頼度つきの誤差上限を計算する補助関数です．

In [6]:
# 悲観的誤差推定を計算する関数
# e: 誤分類数，N: サンプル数，z: z値（デフォルト0.69 ≈ 信頼度75%）
def pessimistic_error(e, N, z=0.69):
    f = e / N  # 誤分類率
    # 悲観的誤差推定の計算式（統計的信頼区間を使った補正）
    return (f + z*z/(2*N) + z * np.sqrt(f/N - f**2/N + z*z/(4*N**2))) / (1 + z*z/N)

# プルーニングを行う関数
# node: 現在のノード，X: 特徴量データ，y: ターゲット変数
def prune(node, X, y):
    if node['is_leaf']:
        return node  # すでにリーフノードならそのまま返す

    # すべての子がリーフノードか確認する
    all_leaf = all(child['is_leaf'] for child in node['children'].values())
    if all_leaf:
        e_subtree = 0  # 部分木の誤分類数
        N_subtree = 0  # 部分木のサンプル数

        # 各子ノードについて誤分類数とサンプル数を計算
        for val, child in node['children'].items():
            mask = X[node['feature']] == val  # 特徴値 val の行を抽出
            y_sub = y[mask]
            e = sum(y_sub != child['majority'])  # 子ノードの誤分類数
            e_subtree += e
            N_subtree += len(y_sub)  # 子ノードのサンプル数

        e_node = sum(y != node['majority'])  # 親ノードの誤分類数
        N_node = len(y)  # 親ノードのサンプル数

        # 悲観的誤差推定を計算
        pe_subtree = pessimistic_error(e_subtree, N_subtree)  # 部分木全体
        pe_node = pessimistic_error(e_node, N_node)  # 親ノード単体

        # 親ノード単体の誤差が小さければ部分木を剪定
        if pe_node <= pe_subtree:
            node['is_leaf'] = True  # 親ノードをリーフノードにする
            node.pop('children', None)  # 子ノードを削除
    else:
        # 子ノードが非リーフノードの場合，再帰的に剪定を続ける
        for val, child in node['children'].items():
            mask = X[node['feature']] == val
            prune(child, X[mask], y[mask])
    return node  # 更新後のノードを返す

#### **■Step 6. 分類木の評価**

predict_one() 関数は，1つのサンプルを入力として木をたどり，リーフノードの多数派クラスを予測します．

evaluate() 関数は，データセット全体に対して predict_one() を繰り返し呼び出し，予測結果と正解ラベルを比較して正解率 (accuracy) を計算します．

In [7]:
# 1つのサンプルに対して分類木をたどり予測を行う関数
# x: サンプル（pandasのSeries），node: 現在のノード情報
def predict_one(x, node):
    while not node['is_leaf']:  # リーフノードに到達するまでループ
        val = x[node['feature']]  # 現在のノードの分割特徴量の値を取得
        if val in node['children']:  # その値に対応する子ノードがある場合
            node = node['children'][val]  # 子ノードに移動
        else:
            break  # 未知の値（学習時に出現しなかった値）の場合はループを抜ける
    return node['majority']  # リーフノードの多数派クラスを返す

# データセット全体に対して分類木の精度を評価する関数
# X: 特徴量データフレーム，y: 正解ラベル，tree: 分類木
def evaluate(X, y, tree):
    # 各サンプルに対して予測を実施
    preds = [predict_one(X.iloc[i], tree) for i in range(len(X))]
    # 予測結果と正解ラベルを比較し，正解率（Accuracy）を計算して返す
    return np.mean(preds == y)

#### **■関数を順に実行して回帰木を構築・評価**

ここまでに定義した関数を順番に呼び出して，分類木の構築・評価を行います．

In [9]:
from collections import Counter

# 実行部分
if __name__ == "__main__":
    X_train, X_test, y_train, y_test = load_data()
    print("\nMethod: C4.5 with Pessimistic Pruning")
    tree = build_tree_c45(X_train, y_train, max_depth=10)
    tree = prune(tree, X_train, y_train)
    acc_train = evaluate(X_train, y_train, tree)
    acc_test = evaluate(X_test, y_test, tree)
    print(f"Train Accuracy: {acc_train:.4f}")
    print(f"Test Accuracy: {acc_test:.4f}")


Method: C4.5 with Pessimistic Pruning
Train Accuracy: 0.8049
Test Accuracy: 0.7762


## **C5.0**

C5.0 は，ID3 や C4.5 と同じく R. Quinlan によって開発された決定木アルゴリズムであり，C4.5 の改良版として位置付けられます．

C4.5 は，情報利得比 (gain ratio) を用いた分割基準，連続値属性の処理，欠損値処理，多値属性の分割などを特徴とし，決定木学習アルゴリズムの代表的存在でした．C5.0 はその枠組みを引き継ぎつつ，理論・実装両面で重要な改良が加えられています．

なお，C5.0 は商用ソフトウェアであり，ソースコードや実装細部は非公開です．そこで以下では，開発者 R. Quinlan による公開論文や公式ドキュメント，および定評のある二次文献，例えば

- J. R. Quinlan, Improved use of continuous attributes in C4.5, *Journal   of Artificial Intelligence Research*, Vol.4, pp.77-90 (1996)

- Ian H. Witten & Eibe Frank, *Data Mining: Practical Machine Learning Tools and Techniques, Second Edition*, Morgan Kaufmann (2005)

- RuleQuest Researchによる公式比較資料, https://www.rulequest.com/see5-comparison.html

などに基づき，公開されている情報の範囲で，C4.5 からの改良点を述べるにとどめます．

<br>

#### **■高速化・メモリ使用量の最適化**

C5.0 は，前身であるC4.5と比較して大幅な高速化とメモリ使用量の削減を実現しています．開発元のベンチマークによれば，条件によって10倍から100倍の高速化が報告されており，大規模データセットへの適応力が向上しました．これは，探索空間の効率的な制御や内部データ構造の最適化によるものとのことです．

<br>

#### **■誤分類コストの導入**

C5.0 の重要な特徴の一つは，クラス間で異なる誤分類コスト（コスト行列）を考慮できる点です．これにより，特定のクラスの誤分類を避けるように学習を調整し，実用性の高い分類器を構築できます．最小化すべき総コストは次の式で表されます．

$$\text{Total_Cost} = \sum_{i,j}^{}{C(i,j)} \cdot N(i,j)$$

ここで，$C(i,j)$ は真のクラス $i$ をクラス $j$ と誤分類した際のコスト，$N(i,j)$ はその誤分類の発生件数です．

<br>

#### **■データ点の重み付け**

各データ点に対して重み (case weights) を割り当てることが可能です．これにより，サンプル数が少ないが重要な観測値や，調査対象の代表性が高いデータを重視して学習させることができます．この重み付けの仕組みは，ブースティングやコスト感度学習の基盤となっています．

<br>

#### **■プルーニング（剪定）の改良**

C4.5 では，信頼区間の上限を用いた悲観的誤差率によるプルーニングを行いました．C5.0 では分割の有意性や信頼性を評価する基準が改良されています．特に，global pruning と呼ばれる，ツリー全体を俯瞰して予測誤差への寄与が低いサブツリーを整理する手法により，精度を維持したまま，より簡潔で汎化性能の高いツリーやルール集合を生成することが可能です．

<br>

#### **■ブースティングの統合**

複数の決定木を逐次的に構築して統合するブースティングアルゴリズムを標準搭載しています．反復回数はパラメータで制御可能であり，適切な設定によって，単一の決定木よりも高い予測精度をもつアンサンブルモデルを構築できます．これは C4.5 からの主要な進化点です．

<br>

#### **■属性選択機能**

winnowing と呼ばれる属性選択機能が組み込まれています．これは学習過程で各属性の予測力を評価し，寄与の小さい属性を自動的に除外する仕組みです．これにより，高次元データやノイズの多いデータセットにおいて，過学習を抑制し学習効率を高める効果があります．

<br>

#### **■マルチスレッドへの対応**

商用実装 (See5/C5.0) においてはマルチスレッド版が提供されており，マルチコア環境での並列計算が可能です．特にブースティングの実行プロセスなどで並列化による高速化が図られています．